# Real gut metagenome (MH0161): LaTeX tables

Builds every table of the real-metagenome analysis as `.tex` (booktabs) from the result files of the pipeline in
`scripts/` (see `README.md`); nothing is recomputed here.

| Content | Input (`results/`) |
|---|---|
| Per tool: reads with a CDS, predicted CDS, CDS with a UniRef90 hit | `diamond_full_cds_per_tool.tsv` |
| Read groups by DeepCDS S / MetaProdigal / FGS (Complete) | `diamond_full_cds_per_cell.tsv` |
| Read groups by DeepCDS N, S, S+I, MetaProdigal, FGS (Complete) | `diamond_full_agreement_all_models.tsv` |
| Read groups by all seven tools | `diamond_full_agreement_all_tools.tsv` |
| Predicted CDS length per tool | `length_distribution_cds.tsv` |

All CDS >= 61 bp (>= 20 aa) on the 58,392,982 reads >= 61 bp. FGS (0.5% / 1.0% error rate) use the indel-corrected
CDS (step 16). A hit is a DIAMOND match against UniRef90 with E <= 1e-3.
Outputs: `results/tables/*.tex` when `SAVE_OUTPUTS = True`.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

ROOT = os.path.abspath(".")
RES = "/tmp/nrt204/FragmentPredictor/real_metagenome_poc/results"
OUT = f"{RES}/tables"
sys.path.insert(0, f"{ROOT}/../eval")
from plot_config import MODEL_DISPLAY_NAMES  # noqa: E402

SAVE_OUTPUTS = True
N_READS = 58_392_982  # reads >= 61 bp
TOOLS = {"DeepCDS N": "deep_cds_no_errors", "DeepCDS S": "deep_cds_substitution", "DeepCDS S+I": "deep_cds_errors",
         "MetaProdigal": "prodigal", "FGS (Complete)": "fgs_complete", "FGS (Illumina 5)": "fgs_illumina_5",
         "FGS (Illumina 10)": "fgs_illumina_10"}
NAMES = {t: MODEL_DISPLAY_NAMES[k] for t, k in TOOLS.items()}  # benchmark display names, e.g. "FGS (0.5% error rate)"
DEEPCDS = ["DeepCDS N", "DeepCDS S", "DeepCDS S+I"]


def tex(s):
    return s.replace("%", r"\%").replace("&", r"\&")


def pct(x, d=1):
    return "--" if pd.isna(x) else f"{100 * x:.{d}f}"


def save_table(lines, name):
    text = "\n".join(lines) + "\n"
    print(text)
    if SAVE_OUTPUTS:
        os.makedirs(OUT, exist_ok=True)
        with open(f"{OUT}/{name}.tex", "w") as fh:
            fh.write(text)


def agreement_table(wide, tools, short, name, caption, label, reads_digits=1, longtable=False):
    """wide: one row per read group (index: group label), columns: reads + one hit rate per tool (NaN = no CDS).
    The highest hit rate in a row is bold."""
    n = len(tools)
    env = "longtable" if longtable else "tabular"
    head = [f"CDS predicted by & Reads (\\%) & " + " & ".join(tex(short[t]) for t in tools) + r" \\"]
    lines = [] if longtable else [r"\begin{table}[ht]", r"\centering", r"\small"]
    if longtable:
        lines += [r"\small", f"\\begin{{longtable}}{{l{'r' * (n + 1)}}}",
                  f"\\caption{{{caption}}}\\label{{{label}}} \\\\", r"\toprule",
                  f" & & \\multicolumn{{{n}}}{{c}}{{CDS with a UniRef90 hit (\\%)}} \\\\", f"\\cmidrule(l){{3-{n + 2}}}",
                  *head, r"\midrule", r"\endfirsthead", r"\toprule", *head, r"\midrule", r"\endhead",
                  r"\bottomrule", r"\endfoot"]
    else:
        lines += [f"\\caption{{{caption}}}", f"\\label{{{label}}}", f"\\begin{{tabular}}{{l{'r' * (n + 1)}}}",
                  r"\toprule", f" & & \\multicolumn{{{n}}}{{c}}{{CDS with a UniRef90 hit (\\%)}} \\\\",
                  f"\\cmidrule(l){{3-{n + 2}}}", *head, r"\midrule"]
    for g, r in wide.iterrows():
        rates = r[tools]
        best = rates.max() if rates.notna().sum() > 1 else np.nan
        cells = [("\\textbf{%s}" % pct(v)) if v == best else pct(v) for v in rates]
        lines.append(f"{tex(g)} & {pct(r['reads'] / N_READS, reads_digits)} & " + " & ".join(cells) + r" \\")
    if longtable:
        lines.append(r"\end{longtable}")
    else:
        lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    save_table(lines, name)


def wide_from_agreement(path, tools):
    d = pd.read_csv(path, sep="\t")
    w = d.pivot_table(index=["n_tools", "group_label", "reads"], columns="tool", values="hit_rate")[tools]
    w = w.sort_index(level=[0, 2], ascending=[False, False]).reset_index(level=[0, 2])
    return w

## Per tool

In [ ]:
t = pd.read_csv(f"{RES}/diamond_full_cds_per_tool.tsv", sep="\t", index_col=0).loc[list(TOOLS)]
t1 = pd.DataFrame({"reads_with_cds": t["reads"] / N_READS, "cds": t["CDS"], "with_hit": t["with_hit"],
                   "without_hit": t["CDS"] - t["with_hit"], "hit_rate": t["hit_rate"]})
display(t1)

best = {"with_hit": t1["with_hit"].max(), "without_hit": t1["without_hit"].min(), "hit_rate": t1["hit_rate"].max()}
bold = lambda s, c, v: f"\\textbf{{{s}}}" if v == best[c] else s
lines = [r"\begin{table}[ht]", r"\centering", r"\small",
         r"\caption{Predicted CDSs on the real gut metagenome sample MH0161 and their support by UniRef90. "
         r"All CDSs $>$60\,bp on the 58.4M reads $>$60\,bp. A CDS has a hit if DIAMOND finds a match with "
         r"E-value $\leq 10^{-3}$. Bold: most CDSs with a hit, fewest without, highest share.}",
         r"\label{tab:real_metagenome_per_tool}",
         r"\begin{tabular}{lrrrrr}", r"\toprule",
         r"Tool & Reads with CDS (\%) & CDSs (M) & With hit (M) & Without hit (M) & CDSs with hit (\%) \\",
         r"\midrule"]
for tool, r in t1.iterrows():
    lines.append(f"{tex(NAMES[tool])} & {pct(r.reads_with_cds)} & {r.cds / 1e6:.2f} & "
                 f"{bold(f'{r.with_hit / 1e6:.2f}', 'with_hit', r.with_hit)} & "
                 f"{bold(f'{r.without_hit / 1e6:.2f}', 'without_hit', r.without_hit)} & "
                 f"{bold(pct(r.hit_rate), 'hit_rate', r.hit_rate)} \\\\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
save_table(lines, "real_metagenome_per_tool")

## Read groups by DeepCDS S, MetaProdigal and FGS (Complete)

Each row is a group of reads, defined by which of the three tools predict a CDS on the read; the numbers are the
share of each tool's CDS on those reads with a hit. `--`: the tool predicts no CDS on these reads.

In [ ]:
cell = pd.read_csv(f"{RES}/diamond_full_cds_per_cell.tsv", sep="\t").set_index(["cell", "tool"])
three = ["DeepCDS S", "MetaProdigal", "FGS (Complete)"]
order = ["all three", "DeepCDS + MetaProdigal", "DeepCDS + FGS", "MetaProdigal + FGS", "DeepCDS alone",
         "MetaProdigal alone", "FGS alone"]
rows = {}
for g in order:
    r = {t: cell.loc[(g, t), "hit_rate"] if (g, t) in cell.index else np.nan for t in three}
    r["reads"] = max(cell.loc[(g, t), "reads"] for t in three if (g, t) in cell.index)
    label = "All three" if g == "all three" else g.replace("DeepCDS", "DeepCDS S")
    rows[label] = r
w2 = pd.DataFrame(rows).T
display(w2)
agreement_table(w2, three, NAMES, "real_metagenome_groups_S_MP_FGS",
                r"Share of predicted CDSs with a UniRef90 hit per read group, defined by which of DeepCDS S, "
                r"MetaProdigal and FGS (Complete) predict a CDS on the read. Reads (\%): share of all 58.4M reads. "
                r"--: the tool predicts no CDS on these reads. Bold: highest share in the row.",
                "tab:real_metagenome_groups_S_MP_FGS")

## Read groups by all DeepCDS models, MetaProdigal and FGS (Complete) (31 groups)

In [ ]:
five = ["DeepCDS N", "DeepCDS S", "DeepCDS S+I", "MetaProdigal", "FGS (Complete)"]
short5 = {"DeepCDS N": "N", "DeepCDS S": "S", "DeepCDS S+I": "S+I", "MetaProdigal": "MP", "FGS (Complete)": "FGS"}
w3 = wide_from_agreement(f"{RES}/diamond_full_agreement_all_models.tsv", five)
display(w3)
agreement_table(w3, five, short5, "real_metagenome_groups_5_tools",
                r"Share of predicted CDSs with a UniRef90 hit per read group, defined by which of DeepCDS N, S, S+I "
                r"(N, S, S+I), MetaProdigal (MP) and FGS (Complete) (FGS) predict a CDS on the read. Reads (\%): "
                r"share of all 58.4M reads. --: the tool predicts no CDS on these reads. Bold: highest share in the row.",
                "tab:real_metagenome_groups_5_tools", reads_digits=2, longtable=True)

## Read groups by all seven tools (120 groups)

FGS-0.5 / FGS-1.0: FGS (0.5% / 1.0% error rate), indel-corrected CDS.

In [ ]:
seven = list(TOOLS)
short7 = {**short5, "FGS (Illumina 5)": "FGS-0.5", "FGS (Illumina 10)": "FGS-1.0"}
w4 = wide_from_agreement(f"{RES}/diamond_full_agreement_all_tools.tsv", seven)
display(w4)
agreement_table(w4, seven, short7, "real_metagenome_groups_7_tools",
                r"Share of predicted CDSs with a UniRef90 hit per read group, defined by which of DeepCDS N, S, S+I "
                r"(N, S, S+I), MetaProdigal (MP), FGS (Complete) (FGS), FGS (0.5\% error rate) (FGS-0.5) and FGS "
                r"(1.0\% error rate) (FGS-1.0) predict a CDS on the read. Reads (\%): share of all 58.4M reads. "
                r"--: the tool predicts no CDS on these reads. Bold: highest share in the row.",
                "tab:real_metagenome_groups_7_tools", reads_digits=3, longtable=True)

# rows where DeepCDS and at least one other tool predict a CDS: does a DeepCDS model have the highest share?
m = w4[seven]
both = m[(m[DEEPCDS].notna().any(axis=1)) & (m.notna().sum(axis=1) > 1)]
print(f"{(both[DEEPCDS].max(axis=1) >= both.max(axis=1)).sum()} of {len(both)} groups: a DeepCDS model is highest")

## Predicted CDS lengths

In [ ]:
L = pd.read_csv(f"{RES}/length_distribution_cds.tsv", sep="\t", index_col=0)[list(TOOLS)]
bins = {r"$\leq$27": L.index <= 27, "28": L.index == 28, "29": L.index == 29, "30": L.index == 30}
t5 = pd.DataFrame({b: L[m].sum() / L.sum() for b, m in bins.items()})
display(t5)
lines = [r"\begin{table}[ht]", r"\centering", r"\small",
         r"\caption{Length of the predicted CDSs (amino acids) on the real gut metagenome (\% of each tool's CDSs). "
         r"99.6\% of the reads are 90\,bp long. FGS leaves the last complete codon of a read out of its "
         r"coordinates and therefore predicts no 30-aa CDS (except after an indel correction).}",
         r"\label{tab:real_metagenome_cds_lengths}",
         r"\begin{tabular}{lrrrr}", r"\toprule",
         r" & \multicolumn{4}{c}{CDS length (aa)} \\", r"\cmidrule(l){2-5}",
         "Tool & " + " & ".join(bins) + r" \\", r"\midrule"]
for tool, r in t5.iterrows():
    lines.append(f"{tex(NAMES[tool])} & " + " & ".join(pct(v) for v in r) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
save_table(lines, "real_metagenome_cds_lengths")